# WEEX V6.48 — MARKET + LIMIT FVG / CONT_SWEEP / REV_MSS
Wspolna historia OI na Google Drive i pobranie ZIP. Strategie LIMIT sa obserwacyjne bez osobnego historycznego potwierdzenia.

In [ ]:
import requests
import shutil
from pathlib import Path
from google.colab import drive, files

# Trwala historia OI, wspolna miedzy wersjami
drive.mount("/content/drive")
oi_dir = Path("/content/drive/MyDrive/WEEX")
oi_dir.mkdir(parents=True, exist_ok=True)
state = oi_dir / "oi_history_shared.json"
if not state.exists():
    poprzednie = [oi_dir / f"weex_v{v}_oi_history_state.json" for v in (647,646,645,644)]
    dostepne = [p for p in poprzednie if p.exists()]
    if dostepne:
        shutil.copy2(max(dostepne, key=lambda p: p.stat().st_mtime), state)

# Pobranie kolektora z GitHub
url = "https://raw.githubusercontent.com/danieldziezak1986-hash/weex-futures/main/weex_v648_collector.py"
r = requests.get(url, timeout=30)
r.raise_for_status()
with open("weex_v648_collector.py", "wb") as f:
    f.write(r.content)

# Testy wbudowane, potem TOP 150 Crypto + TOP 50 TradFi
%run weex_v648_collector.py --self-test
%run weex_v648_collector.py --top 150 --tradfi-top 50 --max-deep 30 --limit-ttl-minutes 180 --state-path /content/drive/MyDrive/WEEX/oi_history_shared.json

# Automatyczne pobranie najnowszego ZIP
folder = Path("/content/weex_v648_results")
archiwa = list(folder.glob("weex_v648_*.zip"))
if not archiwa:
    raise FileNotFoundError("Nie znaleziono ZIP WEEX V6.48")
najnowszy = max(archiwa, key=lambda p: p.stat().st_mtime)
print("Pobieranie wynikow:", najnowszy.name)
files.download(str(najnowszy))
